# Homework 5

# Задача №1 - Можете ли вы отличить сорняки от рассады?

Теперь приступим к задаче классификации на картинках. Реализуйте программу, которая определяет тип рассады на изображении. 

Для того, чтобы определить характерные особенности каждого типа рассады, у вас есть train. Train это папка, в которой картинки уже классифицированы и лежат в соответствующих папках. Исходя из этой информации можете найти признаки, присущие конкретному растению.

Проверка вашего решения будет на происходить на test. В папке test уже нет метки класса для каждой картинки. 

[Ссылка на Яндекс-диск](https://yadi.sk/d/0Zzp0klXT0iRmA), все картинки тут.

Примеры изображений для теста:
<table><tr>
    <td> <img src="https://i.ibb.co/tbqR37m/fhj.png" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="https://i.ibb.co/6yL3Wmt/sfg.png" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="https://i.ibb.co/pvn7NvF/asd.png" alt="Drawing" style="width: 200px;"/> </td>
</tr></table>

In [57]:
import os
import warnings
from pathlib import Path

import cv2
import numpy as np
from matplotlib import pyplot
from tqdm import tqdm

from sklearn import naive_bayes, neighbors, svm, linear_model, tree, ensemble
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score
import sklearn
from catboost import CatBoostClassifier, Pool

In [58]:
class BaseClassifier:
    def __init__(self, model, name):
        self.model = model
        self.model.__name__ = name

    def fit(self, X, Y):
        self.model.fit(X, Y)

    def predict(self, X):
        return self.model.predict(X)

    def plot_decisions(self, X, Y):
        fig = plot_decision_regions(X=X, y=Y, clf=self.model, legend=2)
        pyplot.title(self.__class__.__name__)
        return fig

class NBGaussClassifier(BaseClassifier):
    def __init__(self):
        super().__init__(naive_bayes.GaussianNB(), 'NB Gauss Classifier')

class NBBernoulliClassifier(BaseClassifier):
    def __init__(self):
        super().__init__(naive_bayes.BernoulliNB(), 'NB Bernoulli Classifier')

class LogRegClassifier(BaseClassifier):
    def __init__(self):
        super().__init__(linear_model.LogisticRegression(), 'Logistic Regression Classifier')

class SVMLinearClassifier(BaseClassifier):
    def __init__(self):
        super().__init__(svm.SVC(kernel='linear'), 'SVM linear Classifier')

class SVMPolynomialClassifier(BaseClassifier):
    def __init__(self):
        super().__init__(svm.SVC(kernel='poly'), 'SVM polynomial Classifier')

class SVMRadialClassifier(BaseClassifier):
    def __init__(self):
        super().__init__(svm.SVC(kernel='rbf'), 'SVM radial Classifier')

class KNNClassifier(BaseClassifier):
    def __init__(self):
        super().__init__(neighbors.KNeighborsClassifier(), 'KNN Classifier')

class DecisionTreeClassifier(BaseClassifier):
    def __init__(self):
        super().__init__(tree.DecisionTreeClassifier(), 'Decision Tree Classifier')

class RandomForestClassifier(BaseClassifier):
    def __init__(self):
        super().__init__(ensemble.RandomForestClassifier(), 'Random Forest Classifier')

class GradientBoostingClassifier(BaseClassifier):
    def __init__(self):
        super().__init__(CatBoostClassifier(logging_level='Silent'), 'Gradient Boosting Classifier')
    def fit(self, X, Y):
        train_pool = Pool(X, Y)
        self.model.fit(train_pool)

# Соберем все в одну кучу
classifiers = [cls() for cls in BaseClassifier.__subclasses__()]

In [63]:
# Accuracy - метрика, показывающая долю угаданных ответов из всех возможных ответов
def accuracy(X, y):
    return accuracy_score(X, y)
# # Precision - метрика, которая для одного выбранного класса показывает долю угаданных меток этого класса ко всем угаданным меткам
# def precision_class_0(X, y):
#     return precision_score(X, y, pos_label=[0], average=None)
# def precision_class_1(X, y):
#     return precision_score(X, y, pos_label=1, average=None)
# # Recall - метрика, которая для одного выбранного класса показывает долю угаданных меток этого класса ко всем меткам этого же класса
# def recall_class_0(X, y):
#     return recall_score(X, y, pos_label=0, average=None)
# def recall_class_1(X, y):
#     return recall_score(X, y, pos_label=1, average=None)

metrics = {
    accuracy,
    # precision_class_0,
    # precision_class_1,
    # recall_class_0,
    # recall_class_1,
}

In [64]:
def preprocess(X):
    # Переводим изображение в вектор и изменяем диапазон значений из [0, 255] в [0, 1]
    X = np.reshape(X, (X.__len__(), -1))
    X = X / 255.
    return X


class SVDTransform:
    def __init__(self):
        self.v_T = None

    def transform_forward(self, X, vector_count=500):
        X = preprocess(X)
        print(X.shape)
        # Базисные вектора формируются только для обучающей выборки
        if self.v_T is None:
            u, sigma, v_T = np.linalg.svd(X, full_matrices=False)
            print(u.shape, v_T.shape)
            self.v_T = v_T

        if vector_count == 'all':
            # Раскладывать изображение можно по всем векторам
            return X @ self.v_T.T
        else:
            # А можно только по части векторов (это частично относится к задаче понижения размерности)
            return X @ self.v_T.T[:, :vector_count]

    def transform_backward(self, X):
        # Также напишем обратное преобразование для демонстраций и всего такого
        return np.array([np.sum(x * self.v_T.T[:, :x.shape[0]], axis=1) for x in X])

In [65]:
def read_data(folder: Path, label, target_size=(150, 150)):
    x = [cv2.resize(cv2.imread(imgpath.__str__(), cv2.IMREAD_GRAYSCALE), target_size) for imgpath in folder.glob('*.png')]
    y = [label] * x.__len__()
    return x, y


def calc_metrics(cls, y_pred, y_true):
    metric_vals = {m.__name__: m(y_pred, y_true) for m in metrics}
    print(f'Metrics for {cls.__class__.__name__}: {", ".join([f"{m_}: {val:.3f}" for m_, val in metric_vals.items()])}')
    return metric_vals


def return_best_classifier(X_train, y_train):
    # Делим обучающую выборку на обучающую выборку (поменьше) и валидационную
    X_train_, X_val_, y_train_, y_val_ = train_test_split(X_train, y_train, test_size=0.2, random_state=42, shuffle=True, stratify=y_train)
    # Выберем по какой метрике будем выбирать лучший алгоритм
    reference_metric = 'accuracy'
    max_metric = 0
    best_cls = None
    for cls in classifiers:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            # Обучение
            cls.fit(X_train_, y_train_)
        # Получение ответов на валидационную выборку от обученного на обучающей выборке алгоритма
        y_pred = cls.predict(X_val_)
        # Считаем метрики
        metrics = calc_metrics(cls, y_pred, y_val_)
        # Выбираем лучший алгоритм
        if metrics[reference_metric] > max_metric:
            max_metric = metrics[reference_metric]
            best_cls = cls
    return best_cls

In [69]:
datafolder = Path(os.getcwd()) / 'plants/train'
cls2ind = {'Loose Silky-bent': 0, 'Maize': 1, 'Scentless Mayweed': 2, 'Small-flowered Cranesbill': 3}
ind2cls = {j: i for i, j in cls2ind.items()}

X = []
y = []
for i in range(len(ind2cls)):
    x, y = read_data(datafolder / ind2cls[i], i)
    X += x
    y += y

X = np.array(X)
y = np.array(y)
svd_transform = SVDTransform()
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42, shuffle=True, stratify=Y)

X_train_transformed = svd_transform.transform_forward(X_train, 500)
cls = return_best_classifier(X_train_transformed, y_train)

ValueError: Found input variables with inconsistent numbers of samples: [20, 0]

In [68]:
X_test = read_data(Path(os.getcwd()) / 'plants/test')
X_test_transformed = svd_transform.transform_forward(X_test, 500)
y_pred = cls.predict(X_test_transformed)
calc_metrics(cls, y_pred, y_test);

TypeError: read_data() missing 1 required positional argument: 'label'

# Задача №2 - Собери пазл (2.0).

Даны кусочки изображения, ваша задача склеить пазл в исходную картинку. 

Условия:
* Дано исходное изображение для проверки, использовать собранное изображение в самом алгоритме нельзя;
* Картинки имеют друг с другом пересечение;
* После разрезки кусочки пазлов не были повернуты или отражены;
* НЕЛЬЗЯ выбрать опорную картинку для сбора пазла, как это было в homework 3
* В процессе проверки решения пазлы могут быть перемешаны, т.е. порядок пазлов в проверке может отличаться от исходного 

Изображения расположены по [ссылке](https://disk.yandex.ru/d/XtpawH1sV9UDlg).

Примеры изображений:
<img src="puzzle/su_fighter.jpg" alt="Drawing" style="width: 300px;"/>
<table><tr>
    <td> <img src="puzzle/su_fighter_shuffle/0.jpg" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="puzzle/su_fighter_shuffle/1.jpg" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="puzzle/su_fighter_shuffle/2.jpg" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="puzzle/su_fighter_shuffle/3.jpg" alt="Drawing" style="width: 200px;"/> </td>
</tr></table>

In [ ]:
# Ваш код